# 04 - Census 2022: adults who may vote, and context

Goal: per municipality, the number of adults who may vote (citizens aged 18 and older), the number aged 18 to 29,
and a few rates about services and education.

Reads: `data/raw/census/`
Writes: `INTERIM / "census_2022.csv"`

Run the setup cells first.

## Setup

In [ ]:
import sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Thato20-M/democratic-funnel.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/democratic-funnel")
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "src" / "config.py").exists())

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Repo:", REPO_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repo: /content/democratic-funnel


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from src import config
from src.config import RAW, INTERIM, PROCESSED, MODELS, FIGURES, SEED
from src.io_utils import read_any, clean_columns, inventory
from src.privacy import drop_personal

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings("ignore", category=FutureWarning)

config.make_dirs(); config.set_seed()
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid"); plt.rcParams["figure.dpi"] = 110
print(config.describe())

Environment: Colab
Repo root:   /content/democratic-funnel
Work root:   /content/drive/MyDrive/DIRISA_SDC


## What is in the raw folder

Stats SA files often have extra rows above the real header, so we look at them before we read them.

In [ ]:
inventory(RAW / "census")

,file,sheet,size_kb,header_row,rows,n_cols,columns
0,Census2022sample_F18.csv,None,31229.6,0,1338295,5,QID | Province | District | Municipality | Geo...
1,Census2022sample_F19.csv,None,115318.5,0,1338295,32,QID | DERH_HSIZE | DERH_HHSEX | DERH_HHPOP | D...
2,Census2022sample_F21.csv,None,622838.9,0,4230792,46,QID | PID | P02_SEX | P03_YEAR | P03_MONTH | A...


## 1. Load the raw files

In [ ]:
persons = read_any(RAW / "census" / "Census2022sample_F21.csv")
geo = read_any(RAW / "census" / "Census2022sample_F18.csv")

print(persons.shape)
print(geo.shape)

(4230792, 46)
(1338295, 5)


## 2. Clean the column names and check them

In [ ]:
persons = clean_columns(persons)
geo = clean_columns(geo)

print(persons.columns.tolist())
print(geo.columns.tolist())

['qid', 'pid', 'p02_sex', 'p03_year', 'p03_month', 'age_group', 'p04_age', 'p05_relation', 'p06_marital_st', 'p07a_pop_group', 'p08_language', 'p09_religious_affiliation', 'p10_citizenship', 'p11_prov_pob', 'p12a_countryofbirth', 'derp_birth_region', 'p12b_yearmoved', 'p13a_usualres', 'p14_since2011', 'p15a_resmonthmoved', 'p15b_resyearmoved', 'derp_prevresprov', 'derp_prev_munic', 'derp_usualresprov', 'derp_usualres_munic', 'p16_main_reason', 'p17a_seeing', 'p17b_hearing', 'p17c_communication', 'p17d_walking', 'p17e_remembering', 'p17f_self_care', 'p17a_devmedeyeglas', 'p17b_devmedhearingaid', 'p17c_devmewalkingstick', 'p17e_prosthesis', 'p17d_devmedwheelchair', 'p17f_devmed_other', 'disability_status', 'p18a_motheralive', 'p18b_fatheralive', 'p19_ecd_attendance', 'p20_eduinst', 'p21_edulevel', 'p22_edufield', 'pers_wgt']
['qid', 'province', 'district', 'municipality', 'geo_type']


## 3. Join the person file to geography

The two files share the household number (`qid`).

In [ ]:
cen = persons.merge(geo, on="qid", how="left")

print(cen.shape)
cen.head()

(4230792, 50)


,qid,pid,p02_sex,p03_year,p03_month,age_group,p04_age,p05_relation,p06_marital_st,p07a_pop_group,p08_language,p09_religious_affiliation,p10_citizenship,p11_prov_pob,p12a_countryofbirth,derp_birth_region,p12b_yearmoved,p13a_usualres,p14_since2011,p15a_resmonthmoved,p15b_resyearmoved,derp_prevresprov,derp_prev_munic,derp_usualresprov,derp_usualres_munic,p16_main_reason,p17a_seeing,p17b_hearing,p17c_communication,p17d_walking,p17e_remembering,p17f_self_care,p17a_devmedeyeglas,p17b_devmedhearingaid,p17c_devmewalkingstick,p17e_prosthesis,p17d_devmedwheelchair,p17f_devmed_other,disability_status,p18a_motheralive,p18b_fatheralive,p19_ecd_attendance,p20_eduinst,p21_edulevel,p22_edufield,pers_wgt,province,district,municipality,geo_type
0,10000003,10000003001,2,1977,8,9,44,1,6,1,4,3,101,2,888,1,8888,1,1,88,8888,1,CPT,1,CPT,88,2,1,1,1,1,1,2,2,2,1,2,2,0,2,2,8,1,9,88,21.567356,1,CPT,CPT,1
1,10000003,10000003002,2,2004,8,4,17,3,6,1,4,3,101,1,888,1,8888,1,1,88,8888,1,CPT,1,CPT,88,2,1,1,1,1,1,2,2,2,2,2,2,0,1,1,8,4,10,88,18.902319,1,CPT,CPT,1
2,10000005,10000005001,2,1980,99,9,41,1,1,1,4,1,101,1,888,1,8888,1,1,88,8888,1,CPT,1,CPT,88,1,1,1,1,1,1,1,2,2,9,9,2,0,2,2,8,1,24,10,21.567356,1,CPT,CPT,1
3,10000005,10000005002,1,1978,99,9,43,2,1,1,4,1,101,2,888,1,8888,1,1,88,8888,1,CPT,1,CPT,88,1,1,1,1,1,1,2,2,2,9,9,2,0,2,2,8,6,28,26,23.281103,1,CPT,CPT,1
4,10000005,10000005003,1,2008,99,3,13,3,6,1,4,1,101,1,888,1,8888,1,1,88,8888,1,CPT,1,CPT,88,1,1,1,1,1,1,2,2,2,9,9,2,0,1,1,8,4,7,88,18.902504,1,CPT,CPT,1


## 4. Adults who may vote

Only citizens aged 18 and older can vote, so we count citizens, not all adults. Young adults are citizens aged
18 to 29.

In [ ]:
cen["eligible_adults"] = np.where(
    (cen["p10_citizenship"] == 101) & (cen["p04_age"] >= 18),
    cen["pers_wgt"], 0
)

# Youth = citizens aged 18-29 (inclusive)
cen["eligible_youth"] = np.where(
    (cen["p10_citizenship"] == 101) & (cen["p04_age"].between(18, 29)),
    cen["pers_wgt"], 0
)

print(cen[["p10_citizenship", "p04_age", "pers_wgt", "eligible_adults", "eligible_youth"]].head(10))

# check the national totals before saving anything
youth_total = cen["eligible_youth"].sum()
adult_total = cen["eligible_adults"].sum()
youth_share = youth_total / adult_total
print(f"\neligible_adults: {adult_total:,.0f}")
print(f"eligible_youth:  {youth_total:,.0f}")
print(f"youth / adults:  {youth_share:.3f}")
assert 12e6 <= youth_total <= 13e6, f"eligible_youth = {youth_total:,.0f}, expected 12-13m"
assert 0.31 <= youth_share <= 0.33, f"youth share = {youth_share:.3f}, expected 0.31-0.33"
print("Youth checks passed")

## 5. Check the provincial totals

In [ ]:
pd.set_option("display.float_format", "{:,.0f}".format)
print(cen.groupby("province")["eligible_adults"].sum())

province
1   5,008,537
2   4,546,528
3     879,303
4   1,926,909
5   8,114,913
6   2,395,449
7   9,939,450
8   3,288,050
9   3,951,139
Name: eligible_adults, dtype: float64


## 6. Add up to municipality level

In [ ]:
out = cen.groupby("municipality").agg(
    eligible_adults=("eligible_adults", "sum"),
    eligible_youth=("eligible_youth", "sum"),
).reset_index()

print(out.shape)
out.head()

In [ ]:
# check the municipality table (it is saved at the end)
print(out.shape)
out.head()

## 7. Clean the household file (F19)

In [34]:
house = read_any(RAW / "census" / "Census2022sample_F19.csv")
house = clean_columns(house)

print(house.shape)
print(house.columns.tolist())

(1338295, 32)
['qid', 'derh_hsize', 'derh_hhsex', 'derh_hhpop', 'derh_hhage', 'h01_quarters', 'h02_maindwelling', 'h03_tenure', 'h04_rdp', 'h05_waterpiped', 'h06_watersource', 'h07a_watersupply', 'h08_toilet', 'h09_energy_cooking', 'h10_energy_lighting', 'h11_refuse', 'h12_refrigerator', 'h12_electric_gas_stove', 'h12_vacuum_cleaner', 'h12_washingm', 'h12_computer', 'h12_satellite', 'h12_dvd_player', 'h12_motor_car', 'h12_television', 'h12_radio', 'h12_landline', 'h12_cellphone', 'h13_internet_access', 'a4_adult_hunger', 'a5_child_hunger', 'hh_wgt']


## 8. Join the household file to geography

In [35]:
# join the household file to geography
house_geo = house.merge(geo, on="qid", how="left")

print(house_geo.shape)
house_geo.head()

(1338295, 36)


,qid,derh_hsize,derh_hhsex,derh_hhpop,derh_hhage,h01_quarters,h02_maindwelling,h03_tenure,h04_rdp,h05_waterpiped,h06_watersource,h07a_watersupply,h08_toilet,h09_energy_cooking,h10_energy_lighting,h11_refuse,h12_refrigerator,h12_electric_gas_stove,h12_vacuum_cleaner,h12_washingm,h12_computer,h12_satellite,h12_dvd_player,h12_motor_car,h12_television,h12_radio,h12_landline,h12_cellphone,h13_internet_access,a4_adult_hunger,a5_child_hunger,hh_wgt,province,district,municipality,geo_type
0,10000003,2,2,1,44,5,88,8,8,3,1,2,3,1,1,4,1,2,2,2,2,2,2,2,2,2,2,1,2,1,1,16,1,CPT,CPT,1
1,10000005,4,2,1,41,5,88,8,8,1,1,1,1,1,1,1,1,1,1,1,1,2,2,1,1,1,2,1,1,1,2,16,1,CPT,CPT,1
2,10000008,3,1,2,76,1,1,4,2,1,1,2,1,1,1,1,1,1,2,1,2,1,2,1,1,1,1,1,9,1,1,16,1,DC4,WC043,1
3,10000018,2,2,1,37,1,1,9,9,1,1,9,1,3,1,1,9,9,9,9,9,9,9,9,9,9,9,9,99,9,9,16,1,CPT,CPT,1
4,10000024,1,1,5,34,1,1,4,2,1,1,2,1,1,1,1,1,1,2,1,2,1,2,1,1,2,2,1,2,1,1,16,1,CPT,CPT,1


## 9. Services and education

We turn counts into shares of households: piped water, electricity for lighting and refuse removal. For education
we use the share of citizens aged 25 and older with a qualification after school.

In [38]:
# services, using the Stats SA code list

# piped water: codes 1-6 have some piped water, 7 has none
house_geo["has_piped_water"] = np.where(
    house_geo["h05_waterpiped"].between(1, 6), house_geo["hh_wgt"], 0
)

# electricity for lighting: code 1 is mains electricity
house_geo["has_electricity"] = np.where(
    house_geo["h10_energy_lighting"] == 1, house_geo["hh_wgt"], 0
)

# refuse removal: codes 1-2 are removed by the municipality or a company
house_geo["has_refuse_removal"] = np.where(
    house_geo["h11_refuse"].isin([1, 2]), house_geo["hh_wgt"], 0
)

house_geo["total_hh_weight"] = house_geo["hh_wgt"]

In [39]:
# add up to municipality level and work out the rates
socio = house_geo.groupby("municipality").agg(
    total_households=("total_hh_weight", "sum"),
    piped_water_hh=("has_piped_water", "sum"),
    electricity_hh=("has_electricity", "sum"),
    refuse_removal_hh=("has_refuse_removal", "sum"),
).reset_index()

socio["piped_water_rate"] = socio["piped_water_hh"] / socio["total_households"]
socio["electricity_rate"] = socio["electricity_hh"] / socio["total_households"]
socio["refuse_removal_rate"] = socio["refuse_removal_hh"] / socio["total_households"]

print(socio.shape)
socio.head()

(213, 8)


,municipality,total_households,piped_water_hh,electricity_hh,refuse_removal_hh,piped_water_rate,electricity_rate,refuse_removal_rate
0,BUF,"299,445","289,570","275,467","210,120",1,1,1
1,CPT,"1,589,861","1,562,025","1,499,708","1,427,696",1,1,1
2,EC101,"24,371","24,162","23,190","21,919",1,1,1
3,EC102,"16,631","15,253","15,456","13,087",1,1,1
4,EC104,"31,271","30,395","29,270","27,806",1,1,1


In [40]:
# share with a qualification after school (from the person file)
# P21_EDULEVEL codes 23-27: higher diploma up to PhD
# citizens aged 25+, so students who are still studying are not counted
cen["has_higher_ed"] = np.where(
    (cen["p10_citizenship"] == 101) &
    (cen["p04_age"] >= 25) &
    (cen["p21_edulevel"].between(23, 27)),
    cen["pers_wgt"], 0
)

cen["adults_25plus"] = np.where(
    (cen["p10_citizenship"] == 101) & (cen["p04_age"] >= 25),
    cen["pers_wgt"], 0
)

edu = cen.groupby("municipality").agg(
    higher_ed=("has_higher_ed", "sum"),
    adults_25plus=("adults_25plus", "sum"),
).reset_index()

edu["higher_ed_rate"] = edu["higher_ed"] / edu["adults_25plus"]

In [ ]:
# add the service and education rates to the Census table
out_final = out.merge(socio, on="municipality", how="left")
out_final = out_final.merge(edu[["municipality", "higher_ed_rate"]], on="municipality", how="left")

# check the youth numbers again before saving
youth_total = out_final["eligible_youth"].sum()
youth_share = youth_total / out_final["eligible_adults"].sum()
assert 12e6 <= youth_total <= 13e6, f"eligible_youth = {youth_total:,.0f}, expected 12-13m"
assert 0.31 <= youth_share <= 0.33, f"youth share = {youth_share:.3f}, expected 0.31-0.33"
print(f"eligible_youth: {youth_total:,.0f}  |  youth / adults: {youth_share:.3f}  -> OK")

out_final.to_csv(INTERIM / "census_2022.csv", index=False)
print(out_final.shape)
out_final.head()

In [42]:
print((cen["p04_age"] > 130).sum())

0
